In [ ]:
# Notebook 04 - Limpieza mediante Teoría de Matrices Aleatorias

## Objetivo

Aplicar técnicas de Teoría de Matrices Aleatorias para reducir el ruido presente en la matriz de correlación.

## Entradas

- Matriz de correlación muestral.

## Tareas principales

1. Calcular valores y vectores propios.
2. Determinar los límites de Marčenko-Pastur.
3. Identificar los valores propios asociados al ruido.
4. Aplicar la técnica de clipping.
5. Reconstruir la matriz limpia.

## Salidas

- Matriz de correlación limpia.
- Gráficas del análisis espectral.

## Siguiente paso

La matriz limpia se utiliza en el Notebook 05 para la optimización de portafolios.




# =========================================================
# Notebook 04:
# Análisis de eigenvalores y limpieza RMT
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# =========================================================
# Definir rutas del proyecto
# =========================================================

BASE_DIR = Path(r"C:\MRGB\Personal\Tesis")

OUTPUT_DIR = BASE_DIR / "outputs"

DATOS_PROCESADOS_DIR = OUTPUT_DIR / "datos_procesados"
FIGURAS_DIR = OUTPUT_DIR / "figuras"
MATRICES_DIR = OUTPUT_DIR / "matrices"
TABLAS_DIR = OUTPUT_DIR / "tablas"

FIGURAS_DIR.mkdir(parents=True, exist_ok=True)
MATRICES_DIR.mkdir(parents=True, exist_ok=True)
TABLAS_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# Cargar matriz de covarianza muestral
# =========================================================

archivo_sigma = MATRICES_DIR / "sigma_muestral_80pct.csv"

Sigma = pd.read_csv(archivo_sigma, index_col=0)

Sigma = Sigma.apply(pd.to_numeric, errors="coerce")

print("Dimensiones de Sigma:")
print(Sigma.shape)

print("\nValores faltantes en Sigma:")
print(Sigma.isna().sum().sum())

display(Sigma.head())


# =========================================================
# Descomposición espectral de Sigma
# =========================================================

eigenvalores, eigenvectores = np.linalg.eigh(Sigma)

idx = np.argsort(eigenvalores)[::-1]

eigenvalores = eigenvalores[idx]
eigenvectores = eigenvectores[:, idx]

print("Número de eigenvalores:")
print(len(eigenvalores))

print("\nPrimeros 10 eigenvalores:")
print(eigenvalores[:10])


# =========================================================
# Exportar eigenvalores de covarianza
# =========================================================

tabla_eigenvalores = pd.DataFrame({
    "indice": range(1, len(eigenvalores) + 1),
    "eigenvalor": eigenvalores
})

archivo_eigenvalores = TABLAS_DIR / "eigenvalores_covarianza.csv"

tabla_eigenvalores.to_csv(
    archivo_eigenvalores,
    index=False
)

print("\nEigenvalores guardados en:")
print(archivo_eigenvalores)


# ===========================
# Histograma de eigenvalores
# ===========================

plt.figure(figsize=(10, 6))

plt.hist(eigenvalores, bins=15)

plt.title("Distribución de eigenvalores")
plt.xlabel("Eigenvalor")
plt.ylabel("Frecuencia")

plt.grid(True)

plt.savefig(
    FIGURAS_DIR / "histograma_eigenvalores_covarianza.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =========================================================
# Scree Plot de eigenvalores ordenados
# =========================================================

eigenvalores_ordenados = np.sort(eigenvalores)[::-1]

plt.figure(figsize=(10, 6))

plt.plot(
    range(1, len(eigenvalores_ordenados) + 1),
    eigenvalores_ordenados,
    marker="o"
)

plt.title("Scree Plot de Eigenvalores")
plt.xlabel("Índice")
plt.ylabel("Eigenvalor")

plt.grid(True)

plt.savefig(
    FIGURAS_DIR / "scree_plot_covarianza.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================
# Cálculo del parámetro q = N/T
# ============================================

N = len(eigenvalores)

T = 2644

q = N / T

print("N =", N)
print("T =", T)
print("q =", q)


# ============================================
# Límites de Marčenko-Pastur
# ============================================

lambda_inf = (1 - np.sqrt(q))**2
lambda_sup = (1 + np.sqrt(q))**2

print("lambda_inferior =", lambda_inf)
print("lambda_superior  =", lambda_sup)


# ==================================================
# Cálculo y exportación de la matriz de correlación
# ==================================================

nombres_emisoras = list(Sigma.columns)

matriz_covarianza_np = Sigma.to_numpy(dtype=float)

desviaciones_estandar = np.sqrt(
    np.diag(matriz_covarianza_np)
)

matriz_diagonal_inv_raiz = np.diag(
    1 / desviaciones_estandar
)

matriz_correlacion_np = (
    matriz_diagonal_inv_raiz
    @ matriz_covarianza_np
    @ matriz_diagonal_inv_raiz
)

matriz_correlacion = pd.DataFrame(
    matriz_correlacion_np,
    index=nombres_emisoras,
    columns=nombres_emisoras
)

print("Dimensiones de matriz de correlación:")
print(matriz_correlacion.shape)

print("\nValores faltantes:")
print(matriz_correlacion.isna().sum().sum())

display(matriz_correlacion.head())

# ============================================
# Guardar matriz de correlación
# ============================================

archivo_correlacion = MATRICES_DIR / "matriz_correlacion.csv"

matriz_correlacion.to_csv(
    archivo_correlacion,
    index=True
)

print("\nMatriz de correlación guardada en:")
print(archivo_correlacion)

# =========================================================
# Eigenvalores de la matriz de correlación
# =========================================================

eigenvalores_correlacion, eigenvectores_correlacion = np.linalg.eigh(
    matriz_correlacion
)

# Ordenar de mayor a menor
idx = np.argsort(eigenvalores_correlacion)[::-1]

eigenvalores_correlacion = eigenvalores_correlacion[idx]
eigenvectores_correlacion = eigenvectores_correlacion[:, idx]

print("Número de eigenvalores:")
print(len(eigenvalores_correlacion))

print("\nPrimeros 10 eigenvalores:")
print(eigenvalores_correlacion[:10])

# =========================================================
# Exportar eigenvalores de correlación
# =========================================================

tabla_eigenvalores_correlacion = pd.DataFrame({
    "indice": range(
        1,
        len(eigenvalores_correlacion) + 1
    ),
    "eigenvalor": eigenvalores_correlacion
})

archivo_eigenvalores_correlacion = (
    TABLAS_DIR /
    "eigenvalores_correlacion.csv"
)

tabla_eigenvalores_correlacion.to_csv(
    archivo_eigenvalores_correlacion,
    index=False
)

print("Eigenvalores de correlación guardados en:")
print(archivo_eigenvalores_correlacion)

# =========================================================
# Histograma de eigenvalores de correlación
# =========================================================

plt.figure(figsize=(10,6))

plt.hist(
    eigenvalores_correlacion,
    bins=15,
    color="lightgray",
    edgecolor="black",
    alpha=0.6
)

plt.axvline(
    lambda_inf,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Lambda inferior = {lambda_inf:.3f}"
)

plt.axvline(
    lambda_sup,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Lambda superior = {lambda_sup:.3f}"
)

plt.text(
    lambda_inf,
    plt.ylim()[1]*0.9,
    r'$\lambda_{inf}$',
    rotation=90,
    color="red"
)

plt.text(
    lambda_sup,
    plt.ylim()[1]*0.9,
    r'$\lambda_{sup}$',
    rotation=90,
    color="red"
)

plt.title(
    "Eigenvalores de la matriz de correlación"
)

plt.xlabel("Eigenvalor")
plt.ylabel("Frecuencia")

plt.legend()

archivo_figura = (
    FIGURAS_DIR /
    "histograma_eigenvalores_correlacion_mp.png"
)

plt.savefig(
    archivo_figura,
    dpi=300,
    bbox_inches="tight"
)

print("Figura guardada en:")
print(archivo_figura)

plt.grid(True)

plt.show()

# =========================================================
# Identificar eigenvalores informativos
# =========================================================

eigenvalores_informativos = (
    eigenvalores_correlacion[
        eigenvalores_correlacion > lambda_sup
    ]
)

print("Número de eigenvalores informativos:")
print(len(eigenvalores_informativos))

print("\nEigenvalores informativos:")
print(eigenvalores_informativos)

# ============================================
# Exportar eigenvalores informativos
# ============================================

tabla_eigenvalores_informativos = pd.DataFrame(
    {
        "eigenvalor": eigenvalores_informativos
    }
)

archivo_eigenvalores_informativos = (
    TABLAS_DIR /
    "eigenvalores_informativos.csv"
)

tabla_eigenvalores_informativos.to_csv(
    archivo_eigenvalores_informativos,
    index=False
)

print(
    "Eigenvalores informativos guardados en:"
)

print(
    archivo_eigenvalores_informativos
)


# =========================================================
# Eigenvalores informativos según Marčenko-Pastur
# =========================================================

# Eigenvalores mayores al límite superior

eigenvalores_informativos = (
    eigenvalores_correlacion[
        eigenvalores_correlacion > lambda_sup
    ]
)

print("Número de eigenvalores informativos:")
print(len(eigenvalores_informativos))

print("\nEigenvalores informativos:")
print(eigenvalores_informativos)

# =========================================================
# Eigenvalores de ruido
# =========================================================

eigenvalores_ruido = (
    eigenvalores_correlacion[
        eigenvalores_correlacion <= lambda_sup
    ]
)

print("Número de eigenvalores de ruido:")
print(len(eigenvalores_ruido))

print("\nPrimeros eigenvalores de ruido:")
print(eigenvalores_ruido[:10])

#Exportación de los eigenvalores ruido

tabla_eigenvalores_ruido = pd.DataFrame(
    {
        "eigenvalor": eigenvalores_ruido
    }
)

archivo_eigenvalores_ruido = (
    TABLAS_DIR /
    "eigenvalores_ruido.csv"
)

tabla_eigenvalores_ruido.to_csv(
    archivo_eigenvalores_ruido,
    index=False
)

print(
    "Eigenvalores de ruido guardados en:"
)

print(
    archivo_eigenvalores_ruido
)

# =========================================================
# Eigenvalor promedio del ruido
# =========================================================

lambda_ruido = np.mean(
    eigenvalores_ruido
)

print("Eigenvalor promedio del bulk:")
print(lambda_ruido)

tabla_lambda_ruido = pd.DataFrame(
    {
        "lambda_ruido": [lambda_ruido]
    }
)

archivo_lambda_ruido = (
    TABLAS_DIR /
    "lambda_ruido.csv"
)

tabla_lambda_ruido.to_csv(
    archivo_lambda_ruido,
    index=False
)

print("Lambda bulk guardado en:")
print(archivo_lambda_ruido)

# =========================================================
# Construcción de eigenvalores limpios
# =========================================================

# Crear copia de los eigenvalores originales
eigenvalores_limpios = np.copy(
    eigenvalores_correlacion
)

# Reemplazar eigenvalores de ruido
eigenvalores_limpios[
    eigenvalores_limpios <= lambda_sup
] = lambda_ruido

print("Primeros 10 eigenvalores originales:")
print(eigenvalores_correlacion[:10])

print("\nPrimeros 10 eigenvalores limpios:")
print(eigenvalores_limpios[:10])

# =========================================================
# Exportar eigenvalores limpios
# =========================================================

tabla_eigenvalores_limpios = pd.DataFrame(
    {
        "eigenvalor": eigenvalores_limpios
    }
)

archivo_eigenvalores_limpios = (
    TABLAS_DIR /
    "eigenvalores_limpios.csv"
)

tabla_eigenvalores_limpios.to_csv(
    archivo_eigenvalores_limpios,
    index=False
)

print(
    "Eigenvalores limpios guardados en:"
)

print(
    archivo_eigenvalores_limpios
)

# =========================================================
# Reconstrucción de la matriz de correlación limpia
# =========================================================

# Construir matriz diagonal de eigenvalores limpios
matriz_diagonal_limpia = np.diag(
    eigenvalores_limpios
)

# Reconstruir matriz de correlación limpia
matriz_correlacion_limpia = (
    eigenvectores_correlacion
    @ matriz_diagonal_limpia
    @ eigenvectores_correlacion.T
)

print("Dimensiones:")
print(matriz_correlacion_limpia.shape)

print("\nPrimeras filas:")

display(
    pd.DataFrame(
        matriz_correlacion_limpia
    ).head()
)

# =========================================================
# Renormalizar matriz limpia para que la diagonal sea 1
# =========================================================

# Extraer diagonal de la matriz reconstruida
diagonal_limpia = np.sqrt(
    np.diag(matriz_correlacion_limpia)
)

# Construir matriz diagonal inversa
matriz_normalizacion = np.diag(
    1 / diagonal_limpia
)

# Renormalizar
matriz_correlacion_limpia = (
    matriz_normalizacion
    @ matriz_correlacion_limpia
    @ matriz_normalizacion
)

# Convertir a DataFrame conservando nombres
matriz_correlacion_limpia = pd.DataFrame(
    matriz_correlacion_limpia,
    index=matriz_correlacion.index,
    columns=matriz_correlacion.columns
)

# Verificar diagonal
print("Diagonal mínima:")
print(np.diag(matriz_correlacion_limpia).min())

print("Diagonal máxima:")
print(np.diag(matriz_correlacion_limpia).max())

display(matriz_correlacion_limpia.head())

# =========================================================
# Convertir a DataFrame
# =========================================================

matriz_correlacion_limpia = pd.DataFrame(
    matriz_correlacion_limpia,
    index=matriz_correlacion.index,
    columns=matriz_correlacion.columns
)

display(
    matriz_correlacion_limpia.head()
)

# =========================================================
# Exportar matriz de correlación limpia
# =========================================================

archivo_correlacion_limpia = (
    MATRICES_DIR /
    "matriz_correlacion_limpia.csv"
)

matriz_correlacion_limpia.to_csv(
    archivo_correlacion_limpia
)

print(
    "Matriz de correlación limpia guardada en:"
)

print(
    archivo_correlacion_limpia
)

# =========================================================
# Diferencia entre matrices
# =========================================================

diferencia_matrices = (
    matriz_correlacion_limpia
    - matriz_correlacion
)

print("Máxima diferencia absoluta:")
print(
    np.abs(diferencia_matrices).max().max()
)

print("\nPromedio diferencia absoluta:")
print(
    np.abs(diferencia_matrices).mean().mean()
)

# =========================================================
# Exportar matriz diferencia
# =========================================================

archivo_diferencia = (
    MATRICES_DIR /
    "diferencia_correlacion.csv"
)

diferencia_matrices.to_csv(
    archivo_diferencia
)

print("Matriz diferencia guardada en:")
print(archivo_diferencia)

# =========================================================
# Heatmap de diferencias
# =========================================================

plt.figure(figsize=(10,8))

plt.imshow(
    diferencia_matrices,
    aspect="auto"
)

plt.colorbar(
    label="Diferencia"
)

plt.title(
    "Diferencia entre matriz limpia y matriz original"
)

plt.xlabel("Emisoras")
plt.ylabel("Emisoras")

plt.tight_layout()

# Exportar figura
archivo_heatmap = (
    FIGURAS_DIR /
    "heatmap_diferencia_correlacion.png"
)

plt.savefig(
    archivo_heatmap,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Heatmap guardado en:")
print(archivo_heatmap)

# =========================================================
# Distribución de correlaciones
# Original vs Limpia
# =========================================================

correlaciones_originales = (
    matriz_correlacion.values.flatten()
)

correlaciones_limpias = (
    matriz_correlacion_limpia.values.flatten()
)

plt.figure(figsize=(10,6))

plt.hist(
    correlaciones_originales,
    bins=30,
    alpha=0.5,
    color="gray",
    label="Original"
)

plt.hist(
    correlaciones_limpias,
    bins=30,
    alpha=0.5,
    color="red",
    label="Limpia"
)

plt.title(
    "Distribución de correlaciones"
)

plt.xlabel("Correlación")
plt.ylabel("Frecuencia")

plt.legend()

plt.grid(True)

# ============================================
# Exportar figura
# ============================================

archivo_hist_correlaciones = (
    FIGURAS_DIR /
    "comparacion_correlaciones.png"
)

plt.savefig(
    archivo_hist_correlaciones,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Figura guardada en:"
)

print(
    archivo_hist_correlaciones
)

# Detección de cambios verdaderos

print("Original")
print(correlaciones_originales.mean())
print(correlaciones_originales.std())

print("\nLimpia")
print(correlaciones_limpias.mean())
print(correlaciones_limpias.std())

# =========================================================
# Construcción de Sigma limpia
# =========================================================

# Desviaciones estándar originales
desviaciones_estandar = np.sqrt(
    np.diag(Sigma)
)

# Matriz diagonal D
matriz_diagonal = np.diag(
    desviaciones_estandar
)

# Sigma limpia
sigma_limpia_np = (
    matriz_diagonal
    @ matriz_correlacion_limpia.values
    @ matriz_diagonal
)

# Convertir a DataFrame
sigma_limpia = pd.DataFrame(
    sigma_limpia_np,
    index=Sigma.index,
    columns=Sigma.columns
)

print("Dimensiones de Sigma limpia:")
print(sigma_limpia.shape)

display(
    sigma_limpia.head()
)

# =========================================================
# Exportar Sigma limpia
# =========================================================

archivo_sigma_limpia = (
    MATRICES_DIR /
    "sigma_limpia_80pct.csv"
)

sigma_limpia.to_csv(
    archivo_sigma_limpia
)

print(
    "Sigma limpia guardada en:"
)

print(
    archivo_sigma_limpia
)

# =========================================================
# Comparación Sigma original vs Sigma limpia
# =========================================================

diferencia_sigma = (
    sigma_limpia
    - Sigma
)

print(
    "Máxima diferencia absoluta:"
)

print(
    np.abs(diferencia_sigma).max().max()
)

print(
    "\nPromedio diferencia absoluta:"
)

print(
    np.abs(diferencia_sigma).mean().mean()
)

#sigma_limpia.describe()
#Sigma.describe()